# Data Stream Processing - CapyMOA and River Exploration

In [ ]:
%pip install river
%pip install capymoa

In [3]:
import capymoa
print(capymoa.__version__)

0.11.0


In [61]:
import capymoa.datasets
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve,
    auc as pr_auc_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from capymoa.evaluation import prequential_evaluation
from capymoa.datasets import Covtype
from capymoa.datasets import ElectricityTiny
from capymoa.anomaly import HalfSpaceTrees
from capymoa.classifier import OzaBoost
from capymoa.classifier import KNN
from capymoa.classifier import HoeffdingAdaptiveTree

from river import anomaly, compose, preprocessing, metrics
from river import ensemble, tree, neighbors

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)


# Presentation of the dataset

Covtype

## Anomaly Detection

Remarque:
En détection d'anomalie, la métrique accuracy est trompeuse car une grande majorité de la data, (normalement plus de 90%) ne sont pas des anomalies. la classe "anormale" est très sous représenté par rapport à la classe extèmement majoritaire des observations non anormales.

Il faut utiliser des métriques recall, precision, F1 ROCAUC, PR-AUC pour mieux séparer les observations normales et anormales


## Half Space Trees

### CapyMOA HalfSpaceTree

In [17]:
# Import stream data
stream = Covtype()
schema = stream.get_schema()

learner = HalfSpaceTrees(
    schema,
    window_size=250,
    number_of_trees=25,
    max_depth=15,
    random_seed=42
)

y_true_multiclass = []
y_scores = []

# Getting Datas
while stream.has_more_instances():
    instance = stream.next_instance()
    proba = learner.score_instance(instance)

    y_true_multiclass.append(instance.y_index)
    y_scores.append(proba)

    learner.train(instance)

In [18]:
# Analyzing data classes
print("Distribution of classes in the CovType Dataset:")
class_counts = Counter(y_true_multiclass)
for cls, count in sorted(class_counts.items()):
    print(f"  Class {cls}: {count} ({count/len(y_true_multiclass)*100:.2f}%)")


Distribution of classes in the CovType Dataset:
  Class 0: 211840 (36.46%)
  Class 1: 283301 (48.76%)
  Class 2: 35754 (6.15%)
  Class 3: 2747 (0.47%)
  Class 4: 9493 (1.63%)
  Class 5: 17367 (2.99%)
  Class 6: 20510 (3.53%)


Classes 0 and 1 are really majoritary in the Covtype dataset.

To create a binary classification / anomaly detection problem, I will choose:
- classes 0,1 and 2 as normal (91% of the dataset)
- classes 3,4,5 and 6 as anormal (9% of the dataset)

In [19]:
# Conversion in a binary classification
# Anomalies (1) = Rare classes
# Normal (0) = Majoritary classes

total_instances = len(y_true_multiclass)
anomaly_classes = [cls for cls, count in class_counts.items()
                   if count < total_instances * 0.05]

# Conversion in binary Classification
y_true = [1 if label in anomaly_classes else 0 for label in y_true_multiclass]

In [20]:
# Optimization of the decision threshold with F1
best_f1 = 0
best_threshold = 0.5

for threshold in np.arange(0.1, 1.0, 0.05):
    y_pred_temp = [1 if score >= threshold else 0 for score in y_scores]
    f1_temp = f1_score(y_true, y_pred_temp, zero_division=0)
    if f1_temp > best_f1:
        best_f1 = f1_temp
        best_threshold = threshold

# prediction with the best threshold
y_pred = [1 if score >= best_threshold else 0 for score in y_scores]


In [21]:
# Metrics computation
roc_auc = roc_auc_score(y_true, y_scores)

# PR-AUC
precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_scores)
pr_auc = pr_auc_score(recall_curve, precision_curve)

recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print('-'*30 + ' Results ' + '-'*30)
print(f"ROC AUC:    {roc_auc:.4f}")
print(f"PR-AUC:     {pr_auc:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"F1-Score:   {f1:.4f}")

------------------------------ Results ------------------------------
ROC AUC:    0.7321
PR-AUC:     0.1681
Recall:     0.5737
F1-Score:   0.2902


HST detects “isolated points in the feature space,” not conceptually rare classes.

On the Covtype dataset: $\newline$ 
Classes 3, 4, 5, and 6 are rare but not necessarily scattered in the feature space. $\newline$ 
As a result, HST struggles to separate them from normal classes, leading to many false positives and false negatives. $\newline$ 
Because anomalies represent a very small proportion of the data, the PR-AUC metric is inherently low. $\newline$ 
PR-AUC is sensitive to class imbalance. $\newline$ 
Even if the model correctly detects most anomalies, its precision drops quickly.  $\newline$ 

## River HalfSpace Tree

In [7]:
# Load the data
stream = Covtype()

# Model: River Half Space Trees
model = compose.Pipeline(
    preprocessing.MinMaxScaler(),
    anomaly.HalfSpaceTrees(
        n_trees=25,
        height=15,
        window_size=250,
        seed=42
    )
)

# Evaluation
y_true_multiclass = []
y_scores = []

MAX_INSTANCES = 50_000
count = 0

while stream.has_more_instances() and count<MAX_INSTANCES:
    instance = stream.next_instance()
    x = {f"x{i}": instance.x[i] for i in range(len(instance.x))}
    y = instance.y_index
    count += 1

    score = model.score_one(x)
    y_scores.append(score)
    y_true_multiclass.append(y)

    model.learn_one(x)

covtype.arff: 10.7MB [00:01, 8.89MB/s]                           


In [14]:
# Analysing Data
print("Distribution of classes in the CovType Dataset:")
class_counts = Counter(y_true_multiclass)
for cls, count in sorted(class_counts.items()):
    print(f"  Class {cls}: {count} ({count/len(y_true_multiclass)*100:.2f}%)")

total_instances = len(y_true_multiclass)
anomaly_classes = [cls for cls, count in class_counts.items()
                   if count < total_instances * 0.05]

# Conversion in binary Classification
y_true = [1 if label in anomaly_classes else 0 for label in y_true_multiclass]
# Threshold fixé à 0.35
threshold = 0.35
y_pred = [1 if s >= threshold else 0 for s in y_scores]

Distribution of classes in the CovType Dataset:
  Class 0: 10151 (20.30%)
  Class 1: 28793 (57.59%)
  Class 2: 2160 (4.32%)
  Class 3: 2160 (4.32%)
  Class 4: 2416 (4.83%)
  Class 5: 2160 (4.32%)
  Class 6: 2160 (4.32%)


In [16]:
# Evaluation
roc_auc = roc_auc_score(y_true, y_scores)
precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_scores)
pr_auc = pr_auc_score(recall_curve, precision_curve)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

# Résultats
print("\n" + "-"*30 + " Résultats " + "-"*30)
print(f"Seuil utilisé: {threshold}")
print(f"ROC AUC:    {roc_auc:.4f}")
print(f"PR-AUC:     {pr_auc:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"F1-Score:   {f1:.4f}")
print("-"*75)


------------------------------ Résultats ------------------------------
Seuil utilisé: 0.35
ROC AUC:    0.8133
PR-AUC:     0.4954
Recall:     0.9955
F1-Score:   0.3636
---------------------------------------------------------------------------


## Classifier

# Presentation of the dataset - Electricity

Electricity is a classification problem based on the Australian New South Wales Electricity Market.

- Number of instances: 45,312

- Number of attributes: 8

- Number of classes: 2 (UP, DOWN)

The ElectricityTiny data set was collected from the Australian New South Wales Electricity Market, where prices are not fixed. These prices are affected by demand and supply of the market itself and set every five minutes. The Electricity data set contains 45,312 instances, where class labels identify the changes of the price (2 possible classes: up or down) relative to a moving average of the last 24 hours.

## KNN

 KNN in the streaming setting stores a window of the most recent instances and uses them to classify new instances based on the majority class among the k-nearest neighbors.

## CapyMOA KNN

In [66]:
# hyperparameter to be tested
k_values = [5, 10, 15, 20]
window_sizes = [10, 25, 50, 100]

results = []

for k in k_values:
    for window_size in window_sizes:
        stream = ElectricityTiny()

        # model with specified hyperparameter
        model = KNN(
            schema=stream.get_schema(),
            k=k,
            window_size=window_size
        )

        # Evaluation
        result = prequential_evaluation(
            stream=stream,
            learner=model,
            window_size=1000
        )
        accuracy = result['cumulative'].accuracy()
        results.append({
            'K': k,
            'Window_Size': window_size,
            'Accuracy': accuracy
        })

# Dataframe comparing the results depending on K and windiw_size
df = pd.DataFrame(results)
pivot_df = df.pivot(index='K', columns='Window_Size', values='Accuracy')

print("\n--- Comparaison of model performance in function of window size and K ---")
print(pivot_df.round(4))

# find the best hyperparameter
best_row = df.loc[df['Accuracy'].idxmax()]
print("-"*5 + "Best hyperparameter:" + "-"*5)
print(f"K = {best_row['K']}, Window_Size = {best_row['Window_Size']}, Accuracy = {best_row['Accuracy']:.4f}")


--- Comparaison of model performance in function of window size and K ---
Window_Size   10     25     50     100
K                                     
5            80.1  86.15  89.80  87.50
10           69.0  79.35  87.25  86.55
15           69.0  75.25  83.35  85.70
20           69.0  70.10  82.15  84.80
-----Best hyperparameter:-----
K = 5.0, Window_Size = 50.0, Accuracy = 89.8000


## River KNN

In [73]:
# hyperparameter to be tested
k_values = [5, 10, 15, 20]
results = []

for k in k_values:
    stream = ElectricityTiny()

    # Model
    model = (
        preprocessing.StandardScaler() |
        neighbors.KNNClassifier(
            n_neighbors=k,
            weighted=True
        )
    )

    # Evaluation
    acc = metrics.Accuracy()

    while stream.has_more_instances():
        instance = stream.next_instance()

        # Convert to River format
        x = {f'x{i}': instance.x[i] for i in range(len(instance.x))}
        y = instance.y_index

        # Predict class
        y_pred = model.predict_one(x)

        if y_pred is not None:
            acc.update(y, y_pred)

        # Learn
        model.learn_one(x, y)

    # Stocker le résultat
    accuracy = acc.get()
    results.append({
        'K': k,
        'Accuracy': accuracy
    })

In [75]:
# Result Dataframe
df = pd.DataFrame(results)

print("\n=== Tableau récapitulatif ===")
print(df.to_string(index=False))

# Trouver la meilleure valeur de K
best_row = df.loc[df['Accuracy'].idxmax()]
print(f"Best K: {best_row['K']}, Accuracy = {best_row['Accuracy']:.4f}")


=== Tableau récapitulatif ===
 K  Accuracy
 5  0.840920
10  0.837919
15  0.842421
20  0.832916
Best K: 15.0, Accuracy = 0.8424


Note: CapyMOA’s KNN does not support weighted voting. Each neighbor of the observation x contributes equally to the class vote, which leads to greater instability and higher variance in the model, as the choice of K has a much stronger impact on the final prediction. Moreover, to maximize the performances, window_size has also to be optimized.

River’s KNNClassifier, on the other hand, supports weighted voting. This results in better overall performance and greater stability with respect to the choice of K.

# Trees

## CAPYMOA Hoeffding Adaptive Tree

In [ ]:
stream = ElectricityTiny()
schema = stream.get_schema()

# Explicit parameters for comparison
learner = HoeffdingAdaptiveTree(
    schema,
    random_seed=42
)

results = prequential_evaluation(stream, learner, max_instances=1000)
print(f"CapyMOA HAT Accuracy: {results['cumulative'].accuracy():.2f}")


CapyMOA HAT Accuracy: 84.10


## River HoeffdingAdaptiveTreeClassifier

In [ ]:
# Load the same dataset from CapyMOA
stream = ElectricityTiny()

# Create River's Hoeffding Adaptive Tree Classifier
# Parameters matched to CapyMOA's defaults for fair comparison
model = tree.HoeffdingAdaptiveTreeClassifier(
    seed=42
)

# Create accuracy metric for evaluation
accuracy = metrics.Accuracy()

# Process the stream (prequential evaluation)
instance_count = 0
while stream.has_more_instances() and instance_count < 1000:
    instance = stream.next_instance()

    # Convert CapyMOA instance to River format (dictionary)
    x = {f'x{i}': instance.x[i] for i in range(len(instance.x))}
    y = instance.y_index

    # Predict & learning
    y_pred = model.predict_one(x)
    if y_pred is not None:
        accuracy.update(y, y_pred)

    # Train the model
    model.learn_one(x, y)

    instance_count += 1

print(f"River HAT Accuracy: {accuracy.get() * 100:.2f}")

River HAT Accuracy: 83.48


Hoeffding Adaptive Tree in River and CapyMOA have quite the same performences and are quite equivalent.

### Oza Boost

Streaming Gradient Boosted Trees (SGBT), which is trained using weighted squared loss elicited in XGBoost. SGBT is an ensemble and boosting method which exploits hoeffding adaptativ trees with a replacement strategy to detect and recover from drifts, thus enabling the ensemble to adapt without sacrificing the predictive performance.

CapyMOA Ozaboost

In [ ]:
stream = Covtype()

classifier = OzaBoost(
    stream.get_schema(),
    base_learner="trees.HoeffdingTree",  # default base learner (Hoeffding Tree)
    boosting_iterations=60,              # ensemble size
    use_pure_boost=False,
    random_seed=42
)
results = prequential_evaluation(stream, classifier, max_instances=1000)
print(f"CapyMOA OzaBoost Accuracy: {results['cumulative'].accuracy():.1f}")

CapyMOA OzaBoost Accuracy: 67.7


## River ADWIN Boosting Classifier

ADWIN Boosting Classifier is an online ensemble method that combines multiple adaptive base learners—typically Hoeffding Trees—while using the ADWIN change detector to dynamically adjust model weights and replace weak learners when concept drift is detected.
Unlike a single Hoeffding Adaptive Tree, which adapts internally by growing and pruning nodes, ADWIN Boosting leverages ensemble diversity and adaptive weighting to improve robustness to changing data distributions.

In [ ]:
# Load the same dataset from CapyMOA
stream = Covtype()

# Create River's ADWIN Boosting Classifier (similar to OzaBoost)
# Parameters matched to CapyMOA's OzaBoost for fair comparison
model = ensemble.ADWINBoostingClassifier(
    model=tree.HoeffdingTreeClassifier(),
    n_models=60,                    # same as CapyMOA's boosting_iterations
    seed=42
)

# Create accuracy metric for evaluation
accuracy = metrics.Accuracy()

# Process the stream
instance_count = 0
while stream.has_more_instances() and instance_count < 1000:
    instance = stream.next_instance()

    # Convert CapyMOA instance to River format (dictionary)
    x = {f'x{i}': instance.x[i] for i in range(len(instance.x))}
    y = instance.y_index

    # Predict & learning
    y_pred = model.predict_one(x)
    if y_pred is not None:  # Skip first predictions if model is warming up
        accuracy.update(y, y_pred)

    # Train the model
    model.learn_one(x, y)

    instance_count += 1

print(f"River ADWINBoosting Accuracy: {accuracy.get() * 100:.1f}")

River ADWINBoosting Accuracy: 62.0


The ADWIN-based adaptation mechanism appears to be as effective as ensemble reweighting, highlighting that drift detection and adaptation may play a more critical role than ensemble diversity in this context.$\newline$
However, the results show that CapyMOA’s OzaBoost (ensemble of Hoeffding Trees) and River’s ADWIN Adaptive Tree achieve similar accuracies (62-67%), while both Hoeffding Adaptive Trees reach higher scores (~84%). $\newline$
This suggests that, on this dataset, boosting does not provide a clear advantage over a single adaptive tree.$\newline$